# ✈️ Segmentación Inteligente de Pasajeros para un Programa de Lealtad
**Unidad 5 — Subtema 5.1: K-Means**

## Planteamiento del caso
Una aerolínea quiere lanzar un programa de beneficios (upgrades, salas VIP, descuentos y promociones), pero el presupuesto es limitado y **no puede dar el mismo beneficio a todos**.

Se tienen los registros de **150 pasajeros** durante un año, con **dos variables**:

1. `vuelos_por_anio`: número de vuelos realizados en el año.
2. `gasto_anual_usd`: gasto total en boletos y servicios (USD).

No hay etiquetas: **no sabemos de antemano qué tipos de pasajero existen**. El reto como Ingeniero de Datos es descubrir cuántos perfiles hay y asignar a cada pasajero el beneficio adecuado.

## Paso 1: Generación y exploración inicial de los datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

SEED = 42  # semilla fija = resultados reproducibles
rng = np.random.default_rng(SEED)

# Generamos 150 pasajeros con tres comportamientos ocultos (el modelo NO los conoce)
ocasionales = pd.DataFrame({
    "vuelos_por_anio": rng.normal(3, 1, 70),
    "gasto_anual_usd": rng.normal(900, 250, 70)})
negocios = pd.DataFrame({
    "vuelos_por_anio": rng.normal(28, 5, 45),
    "gasto_anual_usd": rng.normal(14000, 2000, 45)})
vacacionistas = pd.DataFrame({
    "vuelos_por_anio": rng.normal(6, 1.5, 35),
    "gasto_anual_usd": rng.normal(7000, 1200, 35)})

df = pd.concat([ocasionales, negocios, vacacionistas], ignore_index=True)
df["vuelos_por_anio"] = df["vuelos_por_anio"].clip(lower=1).round(0)
df["gasto_anual_usd"] = df["gasto_anual_usd"].clip(lower=100).round(2)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)  # mezclamos
df.index.name = "id_pasajero"

print(df.shape)
df.describe()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["vuelos_por_anio"], df["gasto_anual_usd"], alpha=0.7)
plt.xlabel("Vuelos por año (X)")
plt.ylabel("Gasto anual en USD (Y)")
plt.title("Paso 1: Nube de puntos sin etiquetas")
plt.grid(alpha=0.3)
plt.show()

**Observación:** ¿se ven “islas” o zonas naturales? Describe aquí lo que notas.

## Paso 2: Preprocesamiento y escalamiento
K-Means usa distancias euclidianas. Aquí `gasto_anual_usd` (miles) y `vuelos_por_anio` (decenas) están en escalas muy distintas: sin escalar, **el gasto dominaría el cálculo** y sesgaría los grupos.

Solución: estandarizar (Z-score) para que ambas variables tengan media 0 y desviación estándar 1.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[["vuelos_por_anio", "gasto_anual_usd"]])

print("Media:", X_scaled.mean(axis=0).round(3))
print("Desv. estándar:", X_scaled.std(axis=0).round(3))

## Paso 3: Número óptimo de grupos (k) con Coeficiente de Silueta
Probamos k = 2 hasta k = 6. El coeficiente de silueta (de −1 a +1) mide qué tan similar es un pasajero a su propio grupo (cohesión) frente a los vecinos (separación). **Elegimos el k con el puntaje promedio más alto.**

In [ ]:
k_values = range(2, 7)
scores = []
for k in k_values:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_scaled)
    s = silhouette_score(X_scaled, labels)
    scores.append(s)
    print(f"k = {k} -> silueta = {s:.4f}")

k_opt = list(k_values)[int(np.argmax(scores))]
print(f"\n✅ k óptimo: {k_opt}")

plt.figure(figsize=(6, 4))
plt.plot(list(k_values), scores, marker="o")
plt.axvline(k_opt, color="red", linestyle="--", label=f"k óptimo = {k_opt}")
plt.xlabel("Número de grupos (k)")
plt.ylabel("Silueta promedio")
plt.title("Paso 3: Coeficiente de silueta por k")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Paso 4: Modelo definitivo y asignación de etiquetas
Entrenamos K-Means solo con el k óptimo. Obtenemos los **centroides** (el “pasajero promedio” de cada grupo) y una etiqueta de clúster para cada uno de los 150 pasajeros.

In [ ]:
kmeans = KMeans(n_clusters=k_opt, random_state=SEED, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

print(df["cluster"].value_counts().sort_index())
df.head()

## Paso 5: Perfilamiento de negocio y toma de decisiones
Un número de clúster no le dice nada al gerente de la aerolínea. Calculamos el **promedio real (en unidades originales, sin escalar)** de cada grupo y lo bautizamos según su comportamiento.

In [ ]:
perfiles = df.groupby("cluster")[["vuelos_por_anio", "gasto_anual_usd"]].mean().round(1)
perfiles["pasajeros"] = df["cluster"].value_counts().sort_index()
perfiles

In [ ]:
# Nombrado automático según el comportamiento de cada grupo
nombres, decisiones = {}, {}

c_negocios = perfiles["vuelos_por_anio"].idxmax()
resto = perfiles.drop(index=c_negocios)
c_premium = resto["gasto_anual_usd"].idxmax()
c_ocasional = resto["gasto_anual_usd"].idxmin()

nombres[c_negocios] = "Viajeros de Negocios Frecuentes"
nombres[c_premium] = "Vacacionistas Premium"
nombres[c_ocasional] = "Viajeros Ocasionales"

decisiones[c_negocios] = "Membresía élite: salas VIP, prioridad en abordaje y millas dobles (alta frecuencia y alto gasto)."
decisiones[c_premium] = "Ofertas de upgrade a clase ejecutiva y paquetes vacacionales (pocos vuelos, pero gasto alto por viaje)."
decisiones[c_ocasional] = "Promociones de tarifas bajas y campañas para aumentar la frecuencia de viaje (bajo costo para la aerolínea)."

df["perfil"] = df["cluster"].map(nombres)
perfiles["perfil"] = perfiles.index.map(nombres)
perfiles["decision"] = perfiles.index.map(decisiones)
perfiles[["perfil", "vuelos_por_anio", "gasto_anual_usd", "pasajeros", "decision"]]

## Paso 6: Visualización final de clústeres y centroides
Graficamos de nuevo la nube de puntos, ahora coloreada por perfil y con una estrella en cada centroide (convertido a unidades originales).

In [ ]:
centroides = scaler.inverse_transform(kmeans.cluster_centers_)

plt.figure(figsize=(8, 6))
for c, nombre in nombres.items():
    sub = df[df["cluster"] == c]
    plt.scatter(sub["vuelos_por_anio"], sub["gasto_anual_usd"], alpha=0.7, label=nombre)

plt.scatter(centroides[:, 0], centroides[:, 1], marker="*", s=400,
            c="black", edgecolors="white", linewidths=1.5, label="Centroides")

plt.xlabel("Vuelos por año")
plt.ylabel("Gasto anual en USD")
plt.title("Paso 6: Perfiles de pasajeros y centroides")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Conclusiones
- Escribe cuántos perfiles se encontraron y con qué coeficiente de silueta.
- Explica por qué fue necesario escalar las variables.
- Resume qué beneficio recibe cada perfil y por qué.